# Recommender Systems


# Introduction

## Introduction

[![](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/tools4ds/DS701-Materials-FA26/blob/main/jupyter_notebooks/19-Recommender-Systems.ipynb)

In [ ]:
#| echo: false
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display, Markdown

from slideUtilities import OKABE_ITO

%matplotlib inline

Today, we look at a topic that has become enormously important: recommender systems.



In this lecture, we will:

* Define recommender systems
* Review the challenges they pose
* Discuss two classic methods:
    * Collaborative Filtering
    * Matrix Factorization
* Evaluate a recommender honestly, and meet the problem neither method solves: cold start

The optional module [Recommender Systems II](M6-Recommender-Systems-II.qmd) carries the
latent factors into deep models (DLRM).



This section draws heavily on

* These [slides](http://alex.smola.org/teaching/berkeley2012/slides/8_Recommender.pdf) by Alex Smola
* _Matrix Factorization Techniques for Recommender Systems_, [@koren2009matrix]
* _Collaborative Filtering with Temporal Dynamics_, [@koren2009collaborative]




## What are Recommender Systems?

The concept of recommender systems emerged in the late 1990s / early 2000s as social life moved online:

* online purchasing and commerce
* online discussions and ratings
* social information sharing

In these systems the amount of content was exploding and users were having a hard time finding things they were interested in.

> Users wanted recommendations.

---

Over time, the problem has only gotten worse:

![](figs/L20-netflix-options.png)

---

![](figs/L20-amazon-options.png)

---

An enormous need has emerged for systems to help sort through products, services, and content items.

This often goes by the term __personalization.__

Some examples (rough, recent catalogue sizes):

* Movie recommendation (Netflix ~6.5K movies and shows, YouTube ~14B videos)
* Related product recommendation (Amazon ~600M products)
* Web page ranking (Google >>100B pages)
* Social content filtering (Facebook, Twitter)
* Services (Airbnb, Uber, TripAdvisor)
* News content recommendation (Apple News)
* Priority inbox & spam filtering (Gmail)
* Online dating (Match.com)

---

A more formal view:

* User - requests content
* Objects - instances of content
* Context - ratings, purchases, views, device, location, time, history
* Interface - browser, mobile

![](figs/L20-recsys-abstractly.png)

## Inferring Preferences

Unfortunately, users generally have a hard time __explaining__ what types of
content they prefer.

Some early systems worked by interviewing users to ask what they liked.  Those
systems did not work very well.

A very interesting article about the earliest personalization systems is [User Modeling via Stereotypes](https://www.cs.utexas.edu/users/ear/CogSci.pdf) by Elaine Rich, dating from 1979.


---

Instead, modern systems work by capturing user's opinions about __specific__ items.

This can be done actively:

* When a user is asked to **rate** a movie, product, or experience,

or it can be done passively:

* By noting which items a user **chooses** to purchase (for example).

![](figs/L20-example-data.png)


## Challenges

* The biggest issue is __scalability__: typical data for this problem is huge.
  * Millions of objects
  * 100s of millions of users
* Changing user base
* Changing inventory (movies, stories, goods)
* Available features
* Imbalanced dataset
    * User activity / item reviews are power law distributed

This data is a subset of the data presented in: "From amateurs to connoisseurs:
modeling the evolution of user expertise through online reviews," by J. McAuley
and J. Leskovec. WWW, 2013


## Example

Let's look at a dataset for testing recommender systems consisting of Amazon movie reviews.

The full set is 1.7 million reviews with their text (647 MB). We only need two small
files derived from it, both in `data/amazon-movies/`:

* `review-counts.csv` — how many reviews each user wrote and each movie received,
  stored as a frequency table (`kind`, `reviews`, `count`: "`count` users wrote exactly
  `reviews` reviews");
* `dense-block.csv` — the (user, movie, star rating) triples we will factor later.

In [ ]:
import os

DATA = 'data/amazon-movies'
if not os.path.exists(DATA):  # e.g. on Colab: read the public copy instead
    DATA = 'https://raw.githubusercontent.com/tools4ds/DS701-Materials-FA26/main/data/amazon-movies'

counts = pd.read_csv(f'{DATA}/review-counts.csv')
counts.head()

---

Expanding the frequency table gives one number per user and one per movie:

In [ ]:
def expand(kind):
    rows = counts[counts['kind'] == kind]
    return np.repeat(rows['reviews'].values, rows['count'].values)

reviews_per_user = expand('user')
reviews_per_movie = expand('item')

n_users = len(reviews_per_user)
n_movies = len(reviews_per_movie)
n_reviews = reviews_per_user.sum()
display(Markdown(f'There are:\n\n* {n_reviews:,} reviews\n* {n_movies:,} movies\n* {n_users:,} users'))
display(Markdown(f'There are {n_users * n_movies:,} potential reviews, meaning sparsity of {(n_reviews/(n_users * n_movies)):0.4%}'))

where

$$
\text{sparsity}
= \frac{\text{\# of reviews}}{\text{\# of users} \times \text{\# of movies}}
= \frac{\text{\# of reviews}}{\text{\# of potential reviews}}
$$

## Reviews are Sparse

Only about 0.03% of the possible reviews exist -- 99.97% of the entries are missing.

In [ ]:
display(Markdown(f'There are on average {n_reviews/n_movies:0.1f} reviews per movie' +
     f' and {n_reviews/n_users:0.1f} reviews per user'))

## Sparseness is Skewed

Although on average a movie receives 34 reviews and a user writes 14, __almost all
movies and users have fewer.__ On log–log axes the users' rank–frequency curve is close
to a straight line, the signature of a power law; the movie curve is heavy-tailed too.
(The dataset keeps only users and movies with at least 5 reviews.)

In [ ]:
#| fig-align: center
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, counts_k, noun in [(axes[0], reviews_per_movie, 'Movie'),
                           (axes[1], reviews_per_user, 'User')]:
    mean = counts_k.mean()
    frac_below = np.mean(counts_k < mean)
    ranked = np.sort(counts_k)[::-1]
    ax.loglog(np.arange(1, len(ranked) + 1), ranked, color=OKABE_ITO[5], lw=2,
              label='Number of reviews')
    ax.axhline(mean, color=OKABE_ITO[3], lw=2, ls='--', label=f'Average ({mean:.1f})')
    ax.set_xlabel(f'{noun} rank (most reviewed first)', fontsize=12)
    ax.set_ylabel('Number of reviews', fontsize=12)
    ax.set_title(f'Reviews per {noun.lower()}\n{frac_below:0.0%} of {noun.lower()}s below average',
                 fontsize=13)
    ax.legend(fontsize=11)
plt.tight_layout()

## Objective Function

Ultimately, our goal is to predict the rating that a user would give to an item.

For that, we need to define a loss or objective function.

A typical objective function is root mean square error (RMSE)

$$
\text{RMSE} = \sqrt{\frac{1}{|S|} \sum_{(u,i)\in S} (\hat{r}_{ui} - r_{ui})^2},
$$

where

* $r_{ui}$ is the rating that user $u$ gives to item $i$, and
* $S$ is the set of (user, item) pairs with a known rating.

The simplest predictor is the global mean, $\hat r_{ui} = \mu$ for everyone and
everything. Every method below must beat its RMSE; we will measure it on held-out
ratings at the end.

OK, now we know the problem and the data available.   How can we address the problem?

The earliest method developed is called __collaborative filtering.__

# Collaborative Filtering

## Collaborative Filtering

The central idea of collaborative filtering is that the set of known
recommendations can be considered to be a __bipartite graph.__

![](figs/L20-bipartite.png)

The nodes of the bipartite graph are __users__ ($U$) and __items__ ($V$).

Each edge corresponds to a known rating $r_{ui}.$

---

Then recommendations are formed by traversing or processing the bipartite graph.

![](figs/L20-cf-basic-idea.png)

Joe's three war films were also watched by two other users. The films *they* rated
highly, ranked by how many of them agree, are Joe's recommendations.

There are at least two ways this graph can be used.

---

Two ways to form a rating for the pair $(u, i)$:

1. Using **user-user similarity**:
      * look at users that have similar item preferences to user $u$
      * look at how those users rated item $i$


2. Using **item-item similarity**:
      * look at other items that have been liked by similar users as item $i$
      * look at how user $u$ rated those items


Rule of thumb: <span style="background-color: yellow;">compare along the **short** side of the matrix.</span>

* Few items, many users ⟹ **item–item** (e.g., Netflix: ~6.5K titles, ~280M subscribers).
* Few users, many items ⟹ user–user.


---

Why the short side wins:

* **Size.** With $m$ items the item–item similarity table has $m^2$ entries; with
  $n$ users the user–user table has $n^2$. Netflix: $6{,}500^2 \approx 4\times 10^7$
  versus $(2.8\times 10^8)^2 \approx 8\times 10^{16}$.
* **Support.** Each similarity is estimated from the pairs' co-ratings. Two popular
  titles share thousands of raters; two random subscribers share a handful of titles.
* **Stability.** Item–item similarities change slowly, so they can be precomputed
  offline. That is why Amazon, with tens of millions of customers *and* millions of
  products, chose item–item as well [@linden2003amazon; @sarwar2001item].

## Item-Item CF

For item-item similarity, we'll look at **item-item Collaborative Filtering (CF).**

The questions are:

* How do we judge "similarity" of items?
* How do we form a predicted rating?


---

Here is another view of the ratings graph, this time as a matrix that includes missing entries:

![](figs/L20-u-u-cf-1.png)

---

Let's say we want to predict the value of this unknown rating, user 5's rating of item 1:

![](figs/L20-u-u-cf-2.png)

---

We compare item 1 only with items that **user 5 has rated**: items 3, 4, 5 and 6.

![](figs/L20-u-u-cf-3.png)

---

![](figs/L20-u-u-cf-3.png)

Similarity $s_{1j}$ = Pearson correlation of the two item rows, computed over the
users who rated **both** (defined on the next slides):

| item $j$ | co-raters | $s_{1j}$ |
|---|---|---|
| 3 | users 1, 9, 11 | 0.84 |
| 4 | users 3, 11 | −1.0 |
| 5 | users 3, 6, 11 | −0.87 |
| 6 | users 1, 3, 11 | 1.0 |

We use __$k$-nearest neighbors__ with $k = 2$: the two most similar, items **3 and 6**.


---

These similarity scores tell us how much weight to put on the rating of the other items.

![](figs/L20-u-u-cf-4.png)

User 5 rated item 3 a 2 and item 6 a 3, so

$$
\hat{r}_{u=5,\,i=1} = \frac{s_{13}\, r_{5,3} + s_{16}\, r_{5,6}}{s_{13} + s_{16}}
= \frac{0.84 \cdot 2 + 1.0 \cdot 3}{0.84 + 1.0} = 2.54.
$$

---

The user–user mirror on the same matrix: the users who rated item 1 and share at least
three items with user 5 (with two, a correlation is always $\pm 1$, as for item 4 above)
are users 3 and 11. User 3 correlates positively ($0.87$ over
items 4, 5, 6) and rated item 1 a 3; user 11 correlates negatively ($-0.95$) and is
dropped. User–user CF predicts 3.

![](figs/L20-u-u-cf-1.png)

## Similarity

How should we assess similarity of items?

A reasonable approach is to consider items similar if their ratings are
__correlated__, i.e.,

$$\rho(X,Y) = \frac{E\left[(X-\mu_X)(Y-\mu_Y)\right]}{\sigma_X \sigma_Y}.$$

However, note that two items will not have ratings in the same positions.

![](figs/L20-corr-support.png)

So we want to compute correlation only over the users who rated both the items.

## Example


Let's put the ratings in python lists:

In [ ]:
ratings_item_i = [1, np.nan, np.nan, 5, 5, 3, np.nan, np.nan, np.nan, 4, 2, np.nan, np.nan, np.nan, np.nan, 4, np.nan, 5, 4, 1, np.nan]
ratings_item_j = [np.nan, np.nan, 4, 2, 5, np.nan, np.nan, 1, 2, 5, np.nan, np.nan, 2, np.nan, np.nan, 3, np.nan, np.nan, np.nan, 5, 4]

display(Markdown(f'Ratings for item $i$:\n\n{ratings_item_i}'))
display(Markdown(f'Ratings for item $j$:\n\n{ratings_item_j}'))

## Example, continued


Let's drop the non-common ratings:

In [ ]:
# Keep only the positions where both items have a rating
filtered_ratings_item_i = [rating_i for rating_i, rating_j in zip(ratings_item_i, ratings_item_j) if not np.isnan(rating_i) and not np.isnan(rating_j)]
filtered_ratings_item_j = [rating_j for rating_i, rating_j in zip(ratings_item_i, ratings_item_j) if not np.isnan(rating_i) and not np.isnan(rating_j)]

display(Markdown(f'Common ratings for item $i$: {filtered_ratings_item_i}'))
display(Markdown(f'Common ratings for item $j$: {filtered_ratings_item_j}'))

Now we can compute the Pearson correlation coefficient:

In [ ]:
rho = np.corrcoef(filtered_ratings_item_i, filtered_ratings_item_j)[0,1]
display(Markdown(f'Pearson correlation coefficient: {rho:0.2f}'))

Which is a _moderate negative correlation_, meaning that as item $i$ gets rated higher, item $j$ gets rated lower.

## Checking the Worked Example

The same computation reproduces the similarity table of the item–item example.

In [ ]:
nan = np.nan
R_toy = np.array([            # rows = items 1..6, columns = users 1..12
    [1, nan, 3, nan, nan, 5, nan, nan, 5, nan, 4, nan],
    [nan, nan, 5, 4, nan, nan, 4, nan, nan, 2, 1, 3],
    [2, 4, nan, 1, 2, nan, 3, nan, 4, 3, 5, nan],
    [nan, 2, 4, nan, 5, nan, nan, 4, nan, nan, 2, nan],
    [nan, nan, 4, 3, 4, 2, nan, nan, nan, nan, 2, 5],
    [1, nan, 3, nan, 3, nan, nan, 2, nan, nan, 4, nan]])

user = 4                      # user 5, zero-indexed
for j in np.where(~np.isnan(R_toy[:, user]))[0]:
    both = ~np.isnan(R_toy[0]) & ~np.isnan(R_toy[j])
    s = np.corrcoef(R_toy[0, both], R_toy[j, both])[0, 1]
    raters = ', '.join(str(k + 1) for k in np.where(both)[0])
    print(f'item {j + 1}: co-raters {raters}, s_1{j + 1} = {s:5.2f}')

## Pearson Correlation Coefficient

The Pearson correlation coefficient, often denoted as $r$, is a measure of the
linear correlation between two variables $X$ and $Y$. It quantifies the degree
to which a linear relationship exists between the variables. The value of $r$
ranges from -1 to 1, where:

- $r = 1$ indicates a perfect positive linear relationship,
- $r = -1$ indicates a perfect negative linear relationship,
- $r = 0$ indicates no linear relationship.

The formula for the Pearson correlation coefficient is:

$$
r = \frac{\sum (X_i - \bar{X})(Y_i - \bar{Y})}{\sqrt{\sum (X_i - \bar{X})^2 \sum (Y_i - \bar{Y})^2}}
$$

Where:
- $X_i$ and $Y_i$ are the individual sample points,
- $\bar{X}$ and $\bar{Y}$ are the means of the $X$ and $Y$ samples, respectively.

The Pearson correlation coefficient is sensitive to outliers and assumes that the
relationship between the variables is linear and that the data is normally distributed.


## Similarity for Binary Data

In some cases we will need to work with binary $r_{ui}$.

For example, purchase histories on an e-commerce site, or clicks on an ad.

In this case, an appropriate replacement for Pearson $r$ is the
**Jaccard similarity coefficient** or **Intersection over Union**.

$$
J_{Sim}(\mathbf{x}, \mathbf{y}) = \frac{|\mathbf{x} \cap \mathbf{y}|}{|\mathbf{x} \cup \mathbf{y}|}.
$$

See the lecture on [similarity measures](02-Distances-Timeseries.qmd#norms).

## Improving CF in the presence of bias

One problem with the story so far arises due to __bias__.

* Some items are significantly higher or lower rated
* Some users rate substantially higher or lower in general

These properties interfere with similarity assessment.

Bias correction is crucial for CF recommender systems.

We need to include

* Per-user offset
* Per-item offset
* Global offset

## Representing biases

Hence we need to form a per-item bias of:

$$
b_{ui} = \mu + \alpha_u + \beta_i
$$

where

* $b_{ui}$ is the bias of user $u$ for item $i$.
* $\mu$ is the global average rating across all items and users.
* $\alpha_u$ is the offset of user $u$ and
* $\beta_i$ is the offset of item $i$.

If we gather all these elements together we can form:

* $\boldsymbol{\alpha}$ an $n\times 1$ vector of per-user offsets, and
* $\boldsymbol{\beta}$ an $m\times 1$ vector of per-item offsets.


## Estimating biases


How can we estimate the parameters $\boldsymbol{\alpha}$, $\boldsymbol{\beta}$, and $\mu$?

Let's assume for a minute that we had a fully-dense matrix of ratings $R$.

Recall that each of the $m$ rows of $R$ represents an item and each of the $n$ columns represents a user.

![](figs/L20-u-u-cf-1.png)

## Estimating biases, continued

One way to do this is to minimize the squared error between the ratings and the biases:

$$
\min_{\boldsymbol{\alpha},\boldsymbol{\beta},\mu}
\Vert R - (\mathbf{1}\boldsymbol{\alpha}^T + \boldsymbol{\beta}\mathbf{1}^T + \mu1)\Vert^2
+ \lambda(\Vert\boldsymbol{\alpha}\Vert^2 + \Vert\boldsymbol{\beta}\Vert^2).
$$

and include a regularization term to minimize the magnitude of the biases.

* Here, bold-faced $\mathbf{1}$ represents appropriately sized vectors of ones,
and non-boldfaced $1$ is an $m\times n$ matrix of ones.

* So $\mathbf{1}\boldsymbol{\alpha}^T$ is an $m\times n$ matrix in which each column is constant, equal to that user's offset $\alpha_u$.

* Similarly, $\boldsymbol{\beta}\mathbf{1}^T$ is an $m\times n$ matrix in which each row is constant, equal to that item's offset $\beta_i$.

* And $\mu1$ is an $m\times n$ matrix where each element is $\mu$.


There is a simple strategy for solving it.

Assume we hold $\beta\mathbf{1}^T$ and $\mu1$ constant.

Then the remaining problem is

$$
\min_{\alpha} \Vert R - \mathbf{1}\alpha^T \Vert^2 + \lambda \Vert\alpha\Vert^2,
$$

which (for each column of $R$) is a standard regularized least squares problem
solved via [Ridge regression](./11-Regression-II-Logistic-Regularization.qmd#ridge-regression):
$\hat{\boldsymbol{\beta}} = (X^TX + cI)^{-1}X^T\mathbf{y}$.

## Back to Solving the problem

This problem is __convex__ — in fact jointly convex in $\mu$, $\alpha$ and $\beta$,
because the prediction is linear in all three: it is one big least-squares problem.
Solving for one block at a time is just convenient. (The matrix-factorization problem
below is *not* jointly convex.)

The strategy for solving is:

1. Hold $\alpha$ and $\beta$ constant, solve for $\mu$.
2. Hold $\alpha$ and $\mu$ constant, solve for $\beta$.
3. Hold $\beta$ and $\mu$ constant, solve for $\alpha$.

Each of the three steps will reduce the overall error.  As a result, we iterate over them until convergence.

---

The last issue is that the matrix $R$ is not dense - in reality we only have a small subset of its entries.

We simply need to adapt the least-squares solution to only consider the entries in $R$ that we know.

As a result, the actual calculation is as follows...

---

Step 1:

$$
\mu = \frac{\sum_{(u, i) \in R} (r_{ui} - \alpha_u - \beta_i)}{|R|}
$$

Step 2:

$$
\alpha_u = \frac{\sum_{i \in R(u)}(r_{ui} - \mu - \beta_i)}{\lambda + |R(u)|}
$$

Step 3:

$$
\beta_i = \frac{\sum_{u \in R(i)}(r_{ui} - \mu - \alpha_u)}{\lambda + |R(i)|}
$$

Step 4: If not converged, go to Step 1.

Here $i \in R(u)$ means the set of items rated by user $u$ and $u \in R(i)$ means
the set of users who have rated item $i$ and $|R(u)|$ is the number of ratings.

---

Now that we have learned the biases, we can do a better job of estimating correlation:

$$
\hat{\rho}_{ij} = \frac{\sum_{u\in U(i,j)}(r_{ui} - b_{ui})(r_{uj}-b_{uj})}
{\sqrt{\sum_{u\in U(i,j)}(r_{ui} - b_{ui})^2\sum_{u\in U(i,j)}(r_{uj}-b_{uj})^2}},
$$

where

* $b_{ui} = \mu + \alpha_u + \beta_i$, and
* $U(i,j)$ are the users who have rated both $i$ and $j$.

When $b_{ui}$ is replaced by user $u$'s mean rating, this is the
**adjusted cosine similarity** you will use in the activity.

---

And using biases we can also do a better job of estimating ratings:

$$
\hat{r}_{ui} = b_{ui} + \frac{\sum_{j \in n_k(i, u)} s_{ij}(r_{uj} - b_{uj})}{\sum_{j \in n_k(i, u)} s_{ij}},
$$

where

* $b_{ui} = \mu + \alpha_u + \beta_i$,
* $n_k(i, u)$ are the $k$ nearest neighbors to $i$ that were rated by user $u$ and
* $s_{ij}$ is the similarity between items $i$ and $j$, estimated as above.

## Negative Similarity Scores?

When using correlation coefficient as the similarity score, negative values have important implications for the weighted calculation.

Looking at the weighted rating formula:

$$
\hat{r}_{ui} = b_{ui} + \frac{\sum_{j \in n_k(i, u)} s_{ij}(r_{uj} - b_{uj})}{\sum_{j \in n_k(i, u)} s_{ij}},
$$


## How Negative Similarity Scores Work

When $s_{ij}$ is **negative** (negative correlation), here's what happens:

1. **Numerator**: The term $s_{ij}(r_{uj} - b_{uj})$ becomes negative when $s_{ij} < 0$. This means:
   - If user $u$ rated item $j$ **above** its bias ($r_{uj} - b_{uj} > 0$), the negative similarity will push the prediction for item $i$ **down**
   - If user $u$ rated item $j$ **below** its bias ($r_{uj} - b_{uj} < 0$), the negative similarity will push the prediction for item $i$ **up**

   This makes intuitive sense: if items $i$ and $j$ are negatively correlated, then liking one means you're likely to dislike the other.

2. **Denominator**: When $s_{ij}$ is negative, it contributes a negative value to the sum in the denominator. This can potentially lead to issues:
   - If all similarities are negative, the denominator becomes negative
   - If there's a mix of positive and negative similarities, they might partially cancel out, potentially making the denominator close to zero or even zero

## Common Solutions

To address these issues, practitioners typically use one of these approaches:

1. **Use only positive similarities**: Filter out items with negative correlation (only use $k$-NN where $s_{ij} > 0$)

2. **Use absolute values in denominator**: $\displaystyle\hat{r}_{ui} = b_{ui} + \frac{\sum_{j \in n_k(i, u)} s_{ij}(r_{uj} - b_{uj})}{\sum_{j \in n_k(i, u)} |s_{ij}|}$

The first approach (filtering to positive similarities) is most common because negatively correlated items are typically less informative for prediction than positively correlated items in most domains.

You will sometimes see a third option, rescaling to $[0, 1]$ with $s'_{ij} = (s_{ij} + 1)/2$.
Avoid it: it gives an uncorrelated neighbour weight $0.5$ and an anti-correlated one
weight near $0$, so it rewards neighbours that carry no information.


## Assessing CF

This completes the high level view of CF.

Working with user-user similarities is analogous.

Strengths:

* Essentially no training.
    * The reliance on $k$-nearest neighbors helps in this respect.
* Easy to update with new users, items, and ratings.
* Explainable:
    * "We recommend _Minority Report_ because you liked _Blade Runner_ and _Total Recall._"

Weaknesses:

* Accuracy can be a problem -- resulting in poor recommendations
* Scalability can be a problem -- compute grows (think $k$-NN)

# Matrix Factorization Approaches

## Matrix Factorization

Note that standard CF forces us to consider similarity among items, __or__ among
users, but does not take into account __both.__

Can we use both kinds of similarity simultaneously?

We can't use both the rows and columns of the ratings matrix $R$ at the same
time -- the user and item vectors live in different vector spaces.

---

![](figs/L10-Movie-Latent-Space.png)

[@koren2009matrix]

What we could try to do is find a __single__ vector space in which we represent
__both__ users __and__ items, along with a similarity function, such that:

* users who have similar item ratings are similar in the vector space
* items who have similar user ratings are similar in the vector space
* when a given user highly rates a given item, that user and item are similar in the vector space.


We saw this idea in the SVD lecture:
[Low Rank Defines Latent Factors](12-SVD-Low-Rank.qmd#low-rank-defines-latent-factors).
That lecture ended with "the SVD needs every entry" — here is the fix.

This new vector space is called a __latent__ space, and the user and item
representations are called __latent vectors.__

The same shared latent space is the core of deep-learning recommenders
[@naumov2019deep], covered in the optional module
[Recommender Systems II](M6-Recommender-Systems-II.qmd).


---

Now, however, we are working with a matrix which is only __partially observed.__
That is, we only know __some__ of the entries in the ratings matrix.

Nonetheless, we can imagine a situation like this:

![](figs/L20-mf-1.png)

where we decompose the ratings matrix $R$ into two matrices. Each **row** of the left
factor is an item vector $\mathbf v_i$; each **column** of the right factor is a user
vector $\mathbf u_u$.

We want the product of the two matrices to be as close as possible
__to the known values__ of the ratings matrix.

---

What this setup implies is that our similarity function is the __inner product.__

Which means that to predict an unknown rating, we take the __inner product of
latent vectors:__

![](figs/L20-mf-2.png)

Taking, for example, item 2 (the 2nd row of the item factor) and user 5 (the 5th column of the user factor)...

---

We have

$$
\hat r_{u=5,\,i=2} = \mathbf u_5^T \mathbf v_2 = (-0.5 \cdot -2)+(0.6 \cdot 0.3)+(0.5 \cdot 2.4) = 1.0 + 0.18 + 1.2 = 2.38,
$$

so (rounded):

![](figs/L20-mf-3.png)

## Solving Matrix Factorization

![](figs/L20-mf-1.png)

Notice that in this case we've decided that the factorization should be rank 3,
i.e., low-rank.

So we want something like an SVD.

(Recall that SVD gives us the most-accurate-possible low-rank factorization of a matrix).


However, we can't use the SVD algorithm directly, because we don't know all the entries in $R$.

> Indeed, the unseen entries in $R$ are exactly what we want to predict.

---

Here is what we want to solve:

$$
\min_{U,V} \sum_{(u,i)\in S} \left(r_{ui} - \mathbf u_u^T \mathbf v_i\right)^2 + \lambda(\Vert U\Vert^2 + \Vert V\Vert^2),
$$

or in matrix form $\min_{U,V} \Vert (R - UV^T)_S\Vert^2 + \lambda(\Vert U\Vert^2 + \Vert V\Vert^2)$,

where:

* $U$ is the $n\times k$ **users** matrix: row $u$ is $\mathbf u_u^T$,
* $V$ is the $m\times k$ **items** matrix: row $i$ is $\mathbf v_i^T$,
* $k$ is the rank of the factorization and dimensionality of the latent space, and
* in the matrix form $R$ is laid out users × items ($n \times m$), as in the code
  below; the figure draws its transpose, $R^T \approx VU^T$.

The $(\cdot)_S$ notation means that we are only considering the _subset_ of
matrix entries that correspond to known reviews (the set $S$).

Note that as usual, we add $\ell_2$ penalization to avoid overfitting
([Ridge regression](./11-Regression-II-Logistic-Regularization.qmd#ridge-regression)).

---


This problem is __not__ convex in $U$ and $V$ together, because of the product $UV^T$. It is __biconvex__: convex in $U$ when $V$ is held fixed, and convex in $V$ when $U$ is held fixed.

In particular, if we hold either $U$ or $V$ constant, then the result is a simple
ridge regression.

So one commonly used algorithm for this problem is called __alternating least squares (ALS):__

1. Hold $U$ constant, and solve for $V$
2. Hold $V$ constant, and solve for $U$
3. If not converged, go to Step 1.

## The ALS Update

Hold $V$ fixed. The objective splits into one small problem per user. For user $u$,
let $J_u$ be the items she rated, $V_{J_u}$ their rows of $V$, and $\mathbf r_{u,J_u}$
her ratings:

$$
\min_{\mathbf u_u} \Vert \mathbf r_{u,J_u} - V_{J_u}\mathbf u_u \Vert^2 + \lambda \Vert \mathbf u_u\Vert^2
\quad\Longrightarrow\quad
\mathbf u_u = \left(V_{J_u}^T V_{J_u} + \lambda I\right)^{-1} V_{J_u}^T\, \mathbf r_{u,J_u}.
$$

This is the ridge formula $\hat{\boldsymbol\beta} = (X^TX + cI)^{-1}X^T\mathbf y$ with
$X = V_{J_u}$ (the item vectors are the features), $\mathbf y = \mathbf r_{u,J_u}$
(her ratings) and $c = \lambda$.

The other half-step is the mirror image: for item $i$, regress the ratings it received
on the factors of the users who rated it.

`solve_U` in `scripts/recommender_als.py` is exactly this line, once per user:
`np.linalg.solve(V[W[i]].T.dot(V[W[i]]) + I, V[W[i]].T.dot(x_i[W[i]]))`.

## ALS Update: a Rank-1 Example

Take $k = 1$, so every factor is a number. Fixed item factors:
$v_1 = 1$, $v_2 = 2$, $v_3 = 1.75$.

User $u$ rated $r_{u1} = 2$ and $r_{u2} = 5$, and has not rated item 3.

$$
J(u_u) = (2 - u_u)^2 + (5 - 2u_u)^2 + \lambda u_u^2
\quad\Longrightarrow\quad
u_u = \frac{1\cdot 2 + 2 \cdot 5}{1^2 + 2^2 + \lambda} = \frac{12}{5 + \lambda}.
$$

| | $u_u$ | $\hat r_{u1}$ | $\hat r_{u2}$ | $\hat r_{u3}$ |
|---|---|---|---|---|
| $\lambda = 0$ | 2.4 | 2.4 | 4.8 | 4.2 |
| $\lambda = 1$ | 2.0 | 2.0 | 4.0 | 3.5 |

The penalty pulls the factor toward 0, and every prediction shrinks with it:
**shrinkage**. It costs some fit on the two ratings she gave, and protects against
trusting a factor estimated from very few ratings.

---

The same numbers through the ridge formula:

In [ ]:
V_rated = np.array([[1.0], [2.0]])   # item factors of the items she rated
r_u = np.array([2.0, 5.0])           # her ratings
for lam in [0, 1]:
    u_u = np.linalg.solve(V_rated.T @ V_rated + lam * np.eye(1), V_rated.T @ r_u)
    print(f'lambda = {lam}: u_u = {u_u[0]:.2f}, prediction for item 3 = {1.75 * u_u[0]:.2f}')

## ALS in Practice

The entire Amazon reviews dataset is too large to work with easily, and it is too sparse.

Hence, we will take the densest rows and columns of the matrix: users who wrote more
than 50 reviews and movies with more than 50 reviews. Those star ratings are the file
`dense-block.csv`.

In [ ]:
dense = pd.read_csv(f'{DATA}/dense-block.csv', dtype={'UserId': str, 'ProductId': str})
dense.head()

---

Now we create a $\textnormal{UserID} \times \textnormal{ProductID}$ matrix from these reviews.

Missing entries will be filled with NaNs.

In [ ]:
R = dense.pivot(index='UserId', columns='ProductId', values='Score')
print(f"Review matrix shape: {R.shape}")
print(f"Fraction of entries known: {R.count().sum() / R.size:.2%}")

Even this "dense" block is about 99% empty. A typical $5 \times 4$ corner is all missing:

In [ ]:
R.iloc[900:905, 1000:1004]

---

We'll use code from the [Antidote Data Framework](https://github.com/rastegarpanah/antidote-data-framework) to do the matrix factorization and ALS. We have local copies
`recommender_MF.py`, `recommender_als.py` and `recommender_lmafit.py` in the `scripts/`
folder of our repository.

To evaluate honestly, we first **hide 10% of the known ratings** (per user), fit on the
rest, and score the hidden ones.

In [ ]:
#| code-fold: false
import recommender_MF as MF

np.random.seed(0)
omega_train, omega_val = MF.train_val_split(~R.isnull(), 0.1)
R_train = R.where(omega_train)
print(f'{omega_train.sum().sum():,} training ratings, {omega_val.sum().sum():,} held-out ratings')

---

Fit rank 20 with $\lambda = 1$ on the training ratings:

In [ ]:
#| code-fold: false
RS = MF.als_MF(rank=20, lambda_=1)
%time pred, error = RS.fit_model(R_train)
print(f'Training RMSE: {MF.compute_RMSE(pred, R, omega_train):0.3f}')

The prediction is a dense matrix — every user gets a score for every movie:

In [ ]:
pred.iloc[900:905, 1000:1004].round(2)

## Held-Out Evaluation

In [ ]:
mu = R_train.stack().mean()
baseline = pd.DataFrame(mu, index=R.index, columns=R.columns)
results = [('Global mean $\\mu$', MF.compute_RMSE(baseline, R, omega_train),
            MF.compute_RMSE(baseline, R, omega_val))]
results.append(('MF, rank 20, $\\lambda = 1$', MF.compute_RMSE(pred, R, omega_train),
                MF.compute_RMSE(pred, R, omega_val)))
for rank, lam in [(5, 1), (5, 10)]:
    pred_k, _ = MF.als_MF(rank=rank, lambda_=lam).fit_model(R_train)
    results.append((f'MF, rank {rank}, $\\lambda = {lam}$', MF.compute_RMSE(pred_k, R, omega_train),
                    MF.compute_RMSE(pred_k, R, omega_val)))

table = '| Model | Training RMSE | Held-out RMSE |\n|---|---|---|\n'
table += '\n'.join(f'| {name} | {tr:.3f} | {va:.3f} |' for name, tr, va in results)
display(Markdown(table))

* Rank 20 with $\lambda = 1$ nearly memorises the training ratings — and does **worse
  than the global mean** on ratings it has not seen.
* Fewer factors and a larger penalty give up training fit and win on held-out data.
* $k$ and $\lambda$ are hyperparameters: tune them on a validation split, report a test split.


## What Held-Out RMSE Cannot Tell You

* **Ratings are missing not at random.** People rate what they chose to watch, and
  mostly what they liked. Held-out RMSE is measured on that same biased sample, so it
  flatters popular items and says little about the rest of the catalogue.
* **A recommender ships a list, not a number.** What the product needs is a good top-$k$
  list per user, measured by precision@$k$ or recall@$k$. A model can win on RMSE and
  lose on ranking.
* **Offline ≠ online.** Whether recommendations change what people watch or buy is
  measured with an A/B test on live traffic.

## Assessing Matrix Factorization

Matrix Factorization per se is a good idea.

However, many of the improvements we've discussed for CF apply to MF as well.

To illustrate, we'll look at some of the successive improvements used by the
team that won the Netflix prize ("BellKor's Pragmatic Chaos").

When the prize was announced, the Netflix supplied solution achieved an RMSE of 0.951.

By the end of the competition (about 3 years), the winning team's solution achieved RMSE of 0.856.

Recall our MF objective:

$$
\min_{U, V} \sum_{(u, i)\in S}(r_{ui} - \mathbf u_u^T\mathbf v_i)^2 + \lambda(\Vert U\Vert^2 + \Vert V\Vert^2)
$$

## 1. Adding Biases

If we add biases:
$$
\min_{U, V} \sum_{(u, i)\in S}\left(r_{ui} - (\mu + \alpha_u + \beta_i + \mathbf u_u^T\mathbf v_i)\right)^2
+ \lambda(\Vert U\Vert^2 + \Vert V\Vert^2 + \Vert \alpha\Vert^2 + \Vert \beta \Vert^2)
$$

---

we see improvements in accuracy:

![Matrix factorization models’ accuracy. The plots show the root-mean-square error of each of four individual factor models (lower is better). Accuracy improves when the factor model’s dimensionality (denoted by numbers on the charts) increases. In addition, the more refined factor models, whose descriptions involve more distinct sets of parameters, are more accurate.](figs/L20-netflix-1.png)

## 2. Who Rated What?

In reality, ratings are not provided __at random.__

Take note of which users rated the same movies (ala CF) and use this information.

![](figs/L20-netflix-2.png)

---

![](figs/L20-netflix-3.png)

## 3. Ratings Change Over Time

Older movies tend to get higher ratings!

![](figs/L20-netflix-4.png)

---

If we add time-varying biases, and let each **user's** taste drift (the item
factors stay fixed) [@koren2009collaborative]:

$$
\min_{U, V} \sum_{(u, i)\in S}\left(r_{ui} - (\mu + \alpha_u(t) + \beta_i(t) + \mathbf u_u(t)^T\mathbf v_i)\right)^2
+ \lambda(\Vert U\Vert^2 + \Vert V\Vert^2 + \Vert \alpha\Vert^2 + \Vert \beta \Vert^2)
$$

---

we see further improvements in accuracy:

![](figs/L20-netflix-5.png)

With the time and implicit-feedback terms the model is no longer bilinear in two
blocks of parameters, so the Netflix teams fit it with stochastic gradient descent
([Lecture 16](16-Neural-Networks-I.qmd)) rather than ALS.

## Cold Start

Everything so far needs a **history**. A new user signs up and has rated nothing.

* **CF:** user–user has no co-rated items, so no similarities; item–item has no items
  she rated, so the neighbourhood $n_k(i, u)$ is empty.
* **MF:** no term $(r_{ui} - \mathbf u_u^T\mathbf v_i)^2$ involves her, so only
  $\lambda\Vert\mathbf u_{\text{new}}\Vert^2$ does. It is minimised at
  $\mathbf u_{\text{new}} = \mathbf 0$ (the ALS formula with $J_u$ empty), so
  $\hat r = 0$ for every item. With $\lambda = 0$ her factor is undetermined.
* **MF with biases:** $\alpha_{\text{new}} = 0$ too, so $\hat r_{ui} = \mu + \beta_i$:
  the same popular-items list for everyone. Better, but not personal.
* **New item:** the mirror image. $\mathbf v_{\text{new}} = \mathbf 0$, so it scores 0 for
  everyone and is never recommended.


---

Mitigations — every real system is a **hybrid**:

* **Onboarding:** ask for a few ratings at sign-up ("pick three titles you like").
* **Content and metadata:** genre, cast, description text, or user demographics place
  a new item or user in the latent space before any ratings exist.
* **Popularity prior:** recommend what is popular (or $\mu + \beta_i$) at first.
* **Blend:** weight the prior heavily at first and shift to the collaborative
  prediction as ratings arrive.

## Recap

* Recommender systems predict a sparse, skewed user–item ratings matrix; the global mean is the baseline to beat.
* Collaborative filtering: item–item or user–user $k$-NN with correlation over co-raters, after removing biases; compare along the short side.
* Matrix factorization: users and items in one latent space, $\hat r_{ui} = \mathbf u_u^T\mathbf v_i$, fitted on observed entries only by ALS — one ridge regression per row.
* Evaluate on held-out ratings: training RMSE flatters, and RMSE itself is not the product metric.
* Cold start: with no history, CF and MF fall back to the mean or popularity; hybrids fill the gap.

## References
